# Pump Sensor Data — Data Cleaning
## Dataset
- 220,320 rows of real industrial pump sensor data
- 52 sensor columns (temperatures, pressures, flows, vibrations)
- Source: Kaggle — nphantawee/pump-sensor-data

## Goals
1. Drop unnecessary columns
2. Fix dtypes
3. Investigate duplicates
4. Handle outliers and missing values

## 0. Imports

In [41]:
import pandas as pd
import numpy as np

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.options.display.max_rows = 60

df = pd.read_csv('../data/sensor.csv')

## 1. Drop unnecessary columns
- Drop 'Unnamed: 0' - Original index included in dataset, not needed
- Drop 'sensor_15'- 100% missing values

In [42]:
df = df.drop(['Unnamed: 0', 'sensor_15'], axis=1)

## 2. Fix dtypes
- Convert 'timestamp' from str to datetime
- Convert 'machine_status' from str to category

In [43]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['machine_status'] = df['machine_status'].astype('category')
df[['timestamp', 'machine_status']].dtypes


timestamp         datetime64[us]
machine_status          category
dtype: object

## 3. Investigate duplicates
- 5745 duplicate rows (counting both duplicate rows)
- Determine if pattern is recurring or random
- Decision: drop or keep
- Are duplicate logging artifacts?

Duplicate mask recalculated here rather than persisted - dataset size makes this acceptable.

Further investigating shows the following. There is a pattern of 1 minute and 59 minutes gaps. These occur 2879 and 2844 times respectively. Conclusion is that these are logging artifact. The other gaps are probably real duplicates. Decision is to drop all duplicates. Dropping the remaining 22 rows will have little to no impact.

In [ ]:
# Create mask for duplicate sensor rows
duplicate_rows = df.select_dtypes('float64').duplicated(keep=False)
print(f'Duplicate rows: {duplicate_rows.sum()}')

Duplicate rows: 5745


In [45]:
df.loc[duplicate_rows]['timestamp'].diff()

0                    NaT
1        0 days 00:01:00
60       0 days 00:59:00
61       0 days 00:01:00
120      0 days 00:59:00
               ...      
220141   0 days 00:01:00
220200   0 days 00:59:00
220201   0 days 00:01:00
220260   0 days 00:59:00
220261   0 days 00:01:00
Name: timestamp, Length: 5745, dtype: timedelta64[us]

In [46]:
df.loc[duplicate_rows]['timestamp'].diff().unique()

<TimedeltaArray>
[               NaT,  '0 days 00:01:00',  '0 days 00:59:00',
  '0 days 00:23:00',  '0 days 00:35:00',  '0 days 00:40:00',
  '0 days 00:15:00',  '0 days 00:47:00',  '0 days 00:05:00',
  '5 days 00:15:00', '16 days 09:59:00',  '9 days 04:59:00',
  '3 days 10:59:00',  '0 days 00:12:00',  '0 days 00:46:00',
  '0 days 00:11:00',  '0 days 00:56:00',  '0 days 00:02:00',
  '0 days 00:43:00']
Length: 19, dtype: timedelta64[us]

In [47]:
df.loc[duplicate_rows]['timestamp'].diff().value_counts()

timestamp
0 days 00:01:00     2879
0 days 00:59:00     2844
0 days 00:40:00        2
0 days 00:15:00        2
0 days 00:46:00        2
0 days 00:56:00        2
0 days 00:02:00        2
0 days 00:23:00        1
0 days 00:35:00        1
0 days 00:47:00        1
0 days 00:05:00        1
5 days 00:15:00        1
16 days 09:59:00       1
9 days 04:59:00        1
3 days 10:59:00        1
0 days 00:12:00        1
0 days 00:11:00        1
0 days 00:43:00        1
Name: count, dtype: int64

In [ ]:
# Drop duplicates on sensor columns and verify result
float_cols = df.select_dtypes('float64').columns
df = df.drop_duplicates(subset=float_cols)
duplicate_rows = df.select_dtypes('float64').duplicated(keep=False)
print(f'Duplicate rows: {duplicate_rows.sum()}')
df.shape

Duplicate rows: 0


(217442, 53)

## 4. Handle outliers and missing values

Masking using the column 'machine status' == NORMAL and != NORMAL shows that all the outliers are happening during normal status.
Outliers are therefore process disturbances, sensor failures or sensor range limits. Decided to keep them in the dataset.

Missing values during normal operations are significant. 37% of sensor_50 values are NA and 6% of sensor_51 are NA. Values should be interpolated during follow up.

In [ ]:
# Create mask for machine_status not normal
mask_broken = df['machine_status'] != 'NORMAL'
floats_broken = df.loc[mask_broken].select_dtypes('float64')

# Calculate z-scores > 3 for each sensor
df_zscores_broken = (floats_broken.max() - floats_broken.mean()) / floats_broken.std()
df_zscores_broken.sort_values(ascending=False).loc[df_zscores_broken > 3]

sensor_44   26.82
sensor_13   24.07
sensor_47   23.43
sensor_12   20.36
sensor_45   20.11
sensor_49   18.45
sensor_10   16.86
sensor_41   16.45
sensor_33   15.28
sensor_38   15.09
sensor_48   14.63
sensor_11   13.73
sensor_00   13.20
sensor_50   12.72
sensor_40   11.98
sensor_46   11.75
sensor_43   11.49
sensor_42   10.45
sensor_39    9.71
sensor_18    7.37
sensor_27    7.22
sensor_30    7.09
sensor_16    5.97
sensor_24    5.89
sensor_17    5.12
sensor_32    4.43
sensor_04    4.38
sensor_31    4.11
sensor_01    3.27
sensor_06    3.24
sensor_34    3.16
dtype: float64

In [ ]:
# Create mask for machine_status normal
mask_normal = df['machine_status'] == 'NORMAL'
floats_normal = df.loc[mask_normal].select_dtypes('float64')

# Calculate z-scores > 3 for each sensor
df_zscores_normal = (floats_normal.max() - floats_normal.mean()) / floats_normal.std()
df_zscores_normal.sort_values(ascending=False).loc[df_zscores_normal > 3]

sensor_44   85.63
sensor_39   48.13
sensor_42   47.09
sensor_38   46.64
sensor_41   35.10
sensor_43   34.45
sensor_40   24.69
sensor_49   22.32
sensor_46   21.90
sensor_45   21.53
sensor_50   17.66
sensor_47   17.56
sensor_51   12.84
sensor_27    8.67
sensor_33    7.08
sensor_08    6.89
sensor_09    6.81
sensor_10    5.26
sensor_48    5.12
sensor_07    4.94
sensor_30    4.93
sensor_32    3.98
sensor_29    3.87
sensor_01    3.82
sensor_13    3.44
sensor_04    3.43
sensor_31    3.31
sensor_18    3.26
sensor_28    3.18
sensor_37    3.03
dtype: float64

In [ ]:
# Create mask for NA values in sensor_50 and sensor_51. Count those NA's
missing_normal = df.loc[mask_normal][['sensor_50', 'sensor_51']].isna()
missing_normal.sum()

sensor_50    75642
sensor_51    12168
dtype: int64

In [ ]:
# Create mask for NA values in sensor_50 and sensor_51. Count those NA's
missing_broken = df.loc[mask_broken][['sensor_50', 'sensor_51']].isna()
missing_broken.sum()

sensor_50      79
sensor_51    2949
dtype: int64

In [ ]:
# Calculate percentage of NA's while machine_status = NORMAL for sensor_50
total_s50 = len(df.loc[mask_normal]['sensor_50'])
sum_missing_normal_s50 = missing_normal['sensor_50'].sum()
(sum_missing_normal_s50 / total_s50) * 100

np.float64(37.23107363820268)

In [ ]:
# Calculate percentage of NA's while machine_status = NORMAL for sensor_51
total_s51 = len(df.loc[mask_normal]['sensor_51'])
sum_missing_normal_s51 = missing_normal['sensor_51'].sum()
(sum_missing_normal_s51 / total_s51) * 100

np.float64(5.989102668222022)